# NPS-25-003

Template: Getting_started.ipynb from hepdata_lib

From hepdata_lib:
The following instructions and examples should get you started to get your analysis into [HEPData](https://hepdata.net) using `hepdata_lib`. Please also refer to the [documentation](http://hepdata-lib.readthedocs.io/). While you can also run `hepdata_lib` on your local computer, you can use the [binder](https://mybinder.org/) or [SWAN](http://swan.cern.ch/) services in the browser. Mind that SWAN is only available for people with a CERN account.

Also useful reference: https://github.com/jalimena/HepData_EXO-23-016/tree/main
See "main" function in createHepData_all.py

## General setup

To make sure things are working and `hepdata_lib` is available, run the following command:

In [2]:
import hepdata_lib
import numpy as np
from hepdata_lib import Submission, Table, Variable
from __future__ import print_function
print("hepdata_lib version", hepdata_lib.__version__)

hepdata_lib version 0.20.0


## Adding a table/figure

In HEPData, figures and table will both be `Table` objects. 

The first column is the mass of phi_2, the second of phi_1, and the third is the median upper limit.

Let's create the table/figure. First, we need to give it a name, which is usually just the identifier in the paper, i.e. "Figure _". The table also needs a description, which is usually the caption. You also need to describe the location, i.e. where to find it in the publication:

In [3]:
def make2DLimitTable(tableName, isBDT, fileName, imageName):

    table = Table(tableName)
    if isBDT:
        table.description = "Combined 95% CL observed upper limit on the cross section, using the BDT-based event categorization, as a function of scalar masses."
    else: 
        table.description = "Combined 95% CL observed upper limit on the cross section, using the cut-based event categorization, as a function of scalar masses."
   
    table.location = "Results"
    table.keywords["observables"] = ["SIG"]
    table.keywords["reactions"] = [
        "H -> phi_1 phi_2 -> 2 tau 4 b",
        "H -> phi_1 phi_2 -> 2 tau 2 b"
    ]
    #do I need phrases and "particles"?
    data = np.loadtxt(f"NPS25003_inputs/{fileName}", skiprows=0)

    # Column meaning
    y_vals = data[:, 0]   # FIRST column = y bin centers
    x_vals = data[:, 1]   # SECOND column = x bin centers
    z_vals = data[:, 2]   # bin content

    # Build bin edges from centers
    def make_edges(centers):
        centers = np.unique(centers.astype(float))
        edges = np.zeros(len(centers) + 1)
        edges[1:-1] = 0.5 * (centers[1:] + centers[:-1])
        edges[0] = centers[0] - (edges[1] - centers[0])
        edges[-1] = centers[-1] + (centers[-1] - edges[-2])
        return centers, edges

    y_centers, y_edges = make_edges(y_vals)
    x_centers, x_edges = make_edges(x_vals)

    # Independent variables
    phi2_mass = Variable(
        "phi_2 mass",
        is_independent=True,
        is_binned=True,
        units="GeV"
    )

    phi1_mass = Variable(
        "phi_1 mass",
        is_independent=True,
        is_binned=True,
        units="GeV"
    )

    # Map center -> edge tuple
    y_edges_map = {y: (y_edges[i], y_edges[i+1]) for i, y in enumerate(y_centers)}
    x_edges_map = {x: (x_edges[i], x_edges[i+1]) for i, x in enumerate(x_centers)}

    # Only include bins that exist in your data
    phi2_mass.values = [y_edges_map[y] for y in y_vals]
    phi1_mass.values = [x_edges_map[x] for x in x_vals]

    # Dependent variable
    median_limit = Variable(
        "Median limit",
        is_independent=False,
        is_binned=False,
        units="pb"
    )

    median_limit.values = [float(v) for y,x,v in data] 
    median_limit.add_qualifier("SQRT(S)", "13", "TeV")

    # Add to table
    table.add_variable(phi2_mass)
    table.add_variable(phi1_mass)
    table.add_variable(median_limit)

    table.add_image(f"NPS25003_inputs/{imageName}")
    table.add_additional_resource("Original data file", f"NPS25003_inputs/{fileName}", copy_file=True) #to-do: replace with file and image name
    print(table.name, len(table.variables))
    return table

In [4]:
def make1DLimitTable(tableName, fileName, imageName):
    """
    Reads a 7-column .txt file and creates a single 1D limit Table.
    isBDT and isCascade are inferred from the fileName path:
      - isBDT    = True if 'bdt'   in path, False if 'cutbased' in path
      - isCascade = True if '4b2t' in path, False if '2b2t' in path
    """
    # Infer flags from file path
    isBDT = "bdt" in fileName.lower()
    isCascade = "4b2t" in fileName.lower()

    table = Table(tableName)

    if isCascade:
        table.description = "B(H -> phi_1 phi_2 -> 2 tau 4b) (%)"
    else:
        table.description = "B(H -> phi_1 phi_2 -> 2 tau 2b) (%)"

    table.location = "Results"
    table.keywords["reactions"] = [
        "H -> phi_1 phi_2 -> 2 tau 4 b",
        "H -> phi_1 phi_2 -> 2 tau 2 b"
    ]

    data = np.loadtxt(f"NPS25003_inputs/{fileName}", skiprows=1)

    x_vals  = data[:, 0]
    obs     = data[:, 1]
    exp_m2s = data[:, 2]
    exp_m1s = data[:, 3]
    exp_med = data[:, 4]
    exp_p1s = data[:, 5]
    exp_p2s = data[:, 6]

    def make_edges(centers):
        centers = np.unique(centers.astype(float))
        edges = np.zeros(len(centers) + 1)
        edges[1:-1] = 0.5 * (centers[1:] + centers[:-1])
        edges[0] = centers[0] - (edges[1] - centers[0])
        edges[-1] = centers[-1] + (centers[-1] - edges[-2])
        return centers, edges

    x_centers, x_edges = make_edges(x_vals)
    x_edges_map = {x: (x_edges[i], x_edges[i + 1]) for i, x in enumerate(x_centers)}

    phi1_mass = Variable("phi_1 mass", is_independent=True, is_binned=True, units="GeV")
    phi1_mass.values = [x_edges_map[x] for x in x_vals]

    var_observed = Variable("Observed limit", is_independent=False, is_binned=False, units="pb")
    var_observed.values = [float(v) for v in obs]
    var_observed.add_qualifier("SQRT(S)", "13", "TeV")

    var_exp_med = Variable("Expected limit (median)", is_independent=False, is_binned=False, units="pb")
    var_exp_med.values = [float(v) for v in exp_med]
    var_exp_med.add_qualifier("SQRT(S)", "13", "TeV")

    var_exp_m1s = Variable("Expected limit (-1 sigma)", is_independent=False, is_binned=False, units="pb")
    var_exp_m1s.values = [float(v) for v in exp_m1s]
    var_exp_m1s.add_qualifier("SQRT(S)", "13", "TeV")

    var_exp_p1s = Variable("Expected limit (+1 sigma)", is_independent=False, is_binned=False, units="pb")
    var_exp_p1s.values = [float(v) for v in exp_p1s]
    var_exp_p1s.add_qualifier("SQRT(S)", "13", "TeV")

    var_exp_m2s = Variable("Expected limit (-2 sigma)", is_independent=False, is_binned=False, units="pb")
    var_exp_m2s.values = [float(v) for v in exp_m2s]
    var_exp_m2s.add_qualifier("SQRT(S)", "13", "TeV")

    var_exp_p2s = Variable("Expected limit (+2 sigma)", is_independent=False, is_binned=False, units="pb")
    var_exp_p2s.values = [float(v) for v in exp_p2s]
    var_exp_p2s.add_qualifier("SQRT(S)", "13", "TeV")

    table.add_variable(phi1_mass)
    table.add_variable(var_observed)
    table.add_variable(var_exp_med)
    table.add_variable(var_exp_m1s)
    table.add_variable(var_exp_p1s)
    table.add_variable(var_exp_m2s)
    table.add_variable(var_exp_p2s)

    table.add_image(f"NPS25003_inputs/{imageName}")
    table.add_additional_resource(
        "Original data file",
        f"NPS25003_inputs/{fileName}",
        copy_file=True
    )

    return table

## Main Function

The `Submission` object represents the whole HEPData entry and thus carries the top-level meta data that is equally valid for all the tables and variables you may want to enter. The object is also used to create the physical submission files you will upload to the HEPData web interface.

When using `hepdata_lib` to make an entry, you always need to create a `Submission` object. 

In [13]:
def main():
    submission = Submission()
    submission.read_abstract("NPS25003_inputs/abstract.txt")

    #ADL
    submission.add_additional_resource("ADL file", "NPS25003_inputs/NPS25003.adl", copy_file=True)

    #Production cross section
    submission.add_link("tandard ggF and VBF cross-sections from Handbook of LHC Cross-sections", "http://arxiv.org/abs/arXiv:1610.07922") 

    #Pythia configurations - not necessary since standard

    #Signal model UFO Files
    submission.add_link("Signal Model UFO files", "https://gitlab.com/apapaefs/twosinglet")

    #Generator Process cards
    submission.add_link("Generator Process Cards", "https://github.com/cms-sw/genproductions/pull/2705") 

    #Cut flow tables

    #Data distributions of relevant ML input features

    #Small set of input vectors & ML outputs
    submission.add_link("BDT Models", "https://github.com/Aaravind96/aabbttBDT/tree/preservation/BDTmodels")
    
    #Signal Efficiencies for simplified models' model points

    #Statistical model
    submission.add_link("Datacards", "https://gitlab.cern.ch/cms-analysis/nps/nps-25-003/datacards/-/tree/master/input?ref_type=heads") 
    
    ##################
    # 2D Limit plots #
    ##################
    version_folder = "v11_limits"
    plots = "NPS-25-003/plots"

    table_configs_2D = [
        # (name,                  isBDT, subfolder,    channel)
        ("BDT_allchannels",       True,  "bdt_based",   "allchannels"),
        ("BDT_emu",               True,  "bdt_based",   "emu"),
        ("BDT_mutau",             True,  "bdt_based",   "mutau"),
        ("BDT_etau",              True,  "bdt_based",   "etau"),
        ("cutbased_allchannels",  False, "cut_based",   "allchannels"),
        ("cutbased_emu",          False, "cut_based",   "emu"),
        ("cutbased_mutau",        False, "cut_based",   "mutau"),
        ("cutbased_etau",         False, "cut_based",   "etau"),
    ]

    for name, isBDT, subfolder, channel in table_configs_2D:
        prefix = "bdt_" if isBDT else ""
        submission.add_table(make2DLimitTable(
            name,
            isBDT,
            f"{version_folder}/{subfolder}/median_limits_{channel}.txt",
            f"{plots}/plotLimit_{prefix}2d_{channel}.pdf" 
        ))
        
    ##################
    # 1D Limit plots #
    ##################

    table_configs_1D = [
        # (name,                           subfolder,        decay,   m1)
        ("BDT_4b2t_m1_15",                "bdtbased_root",  "4b2t",  15),
        ("BDT_4b2t_m1_20",                "bdtbased_root",  "4b2t",  20),
        ("BDT_4b2t_m1_30",                "bdtbased_root",  "4b2t",  30),
        ("BDT_2b2t_m1_15",                "bdtbased_root",  "2b2t",  15),
        ("BDT_2b2t_m1_20",                "bdtbased_root",  "2b2t",  20),
        ("BDT_2b2t_m1_30",                "bdtbased_root",  "2b2t",  30),
        ("BDT_2b2t_m1_40",                "bdtbased_root",  "2b2t",  40),
        ("BDT_2b2t_m1_50",                "bdtbased_root",  "2b2t",  50),
        ("cutbased_4b2t_m1_15",           "cutbased_root",  "4b2t",  15),
        ("cutbased_4b2t_m1_20",           "cutbased_root",  "4b2t",  20),
        ("cutbased_4b2t_m1_30",           "cutbased_root",  "4b2t",  30),
        ("cutbased_2b2t_m1_15",           "cutbased_root",  "2b2t",  15),
        ("cutbased_2b2t_m1_20",           "cutbased_root",  "2b2t",  20),
        ("cutbased_2b2t_m1_30",           "cutbased_root",  "2b2t",  30),
        ("cutbased_2b2t_m1_40",           "cutbased_root",  "2b2t",  40),
        ("cutbased_2b2t_m1_50",           "cutbased_root",  "2b2t",  50),
    ]

    for name, subfolder, decay, m1 in table_configs_1D:
        prefix = "bdt_" if "bdt" in subfolder else ""
        fileName = f"{subfolder}/higgsCombine_a1a2_{decay}_allchannels_allyears_m1_{m1}_limits.txt"
        imageName = f"{plots}/plotLimit_{prefix}allyears_a1a2_{decay}_m1_{m1}_allchannels.pdf" 
        submission.add_table(make1DLimitTable(name, fileName, imageName))

    for table in submission.tables:
        table.keywords["cmenergies"] = [13000]
    outdir = "NPS25003_output"
    print("Tables:", [t.name for t in submission.tables])
    submission.create_files(outdir, remove_old=True)

In [ ]:
if __name__ == "__main__":
    main()

BDT_allchannels 3
BDT_emu 3
BDT_mutau 3
BDT_etau 3
cutbased_allchannels 3
cutbased_emu 3
cutbased_mutau 3
cutbased_etau 3
Tables: ['BDT_allchannels', 'BDT_emu', 'BDT_mutau', 'BDT_etau', 'cutbased_allchannels', 'cutbased_emu', 'cutbased_mutau', 'cutbased_etau', 'BDT_4b2t_m1_15', 'BDT_4b2t_m1_20', 'BDT_4b2t_m1_30', 'BDT_2b2t_m1_15', 'BDT_2b2t_m1_20', 'BDT_2b2t_m1_30', 'BDT_2b2t_m1_40', 'BDT_2b2t_m1_50', 'cutbased_4b2t_m1_15', 'cutbased_4b2t_m1_20', 'cutbased_4b2t_m1_30', 'cutbased_2b2t_m1_15', 'cutbased_2b2t_m1_20', 'cutbased_2b2t_m1_30', 'cutbased_2b2t_m1_40', 'cutbased_2b2t_m1_50']


In [ ]:
!cat NPS25003_output/submission.yaml

In [ ]:
!ls NPS25003_output

In [ ]:
!ls submission.tar.gz